# Hotel Booking Optimization
## Dynamic Pricing & Stochastic Overbooking

**Author:** Group One for Artificial Intelligence II  
**Dataset:** Hotel Booking Demand — Antonio, Almeida & Nunes (2019)

---

### What This Notebook Does

The classification notebook tells us *whether* a booking will be cancelled.  
This notebook uses that information to answer a more actionable question: **what should the hotel actually do with it?**

We build two optimization models:

| Model | Question it answers |
|---|---|
| **ADR Regression** | What rate should we charge for this booking? |
| **Stochastic Overbooking Optimizer** | How many extra bookings should we accept to maximize revenue? |

### How They Connect

```
Booking features
      │
      ├──► Cancellation Classifier ──► P(cancel) ──► Overbooking Optimizer ──► k* rooms
      │
      └──► ADR Regressor ──────────────────────────────────────────────────► Optimal rate
```

This notebook is self-contained — it re-runs preprocessing and model training from scratch.

## 1. Setup & Imports

We load all libraries upfront. The key addition here compared to the classification notebook is `scipy.stats.norm`, which we use to compute the inverse CDF (quantile function) needed for the Newsvendor formula in the overbooking optimizer.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    roc_auc_score, mean_squared_error, r2_score
)
from lightgbm import LGBMClassifier, LGBMRegressor
from scipy.stats import norm

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Libraries loaded.')

## 2. Data Loading

We load the same hotel bookings dataset used in the classification notebook.  
It contains 119,390 reservations across two hotel types — City Hotel and Resort Hotel — with 32 original features per booking.

In [ ]:
df = pd.read_csv('hotel_bookings.csv')
print(f'Dataset shape: {df.shape}')
print(f'Cancellation rate: {df["is_canceled"].mean()*100:.1f}%')
print(f'Mean ADR: ${df["adr"].mean():.2f}  |  Median ADR: ${df["adr"].median():.2f}')

## 3. Preprocessing

We apply the same preprocessing decisions as in the classification notebook:

- **Drop post-hoc columns** (`reservation_status`, `reservation_status_date`) — these are only known after the stay ends, so including them would let the model see the future
- **Fill agent/company NaN with 0** — a missing value here means "no agent" or "no company", which is a meaningful distinction
- **Group rare countries** — keep the top 20 by frequency, collapse the rest to `'Other'` to avoid an explosion of sparse one-hot columns
- **One-hot encode nominal columns** — features like hotel type, meal, and market segment have no numeric ordering, so we create binary indicator columns for each category

One important addition: we **preserve `adr` in a separate variable** before dropping it from the feature matrix, since we'll use it as the regression target in Section 4.

In [ ]:
data = df.copy()

# Drop columns that are only known after the event (post-hoc)
data.drop(columns=['reservation_status', 'reservation_status_date'], inplace=True)

# Drop the 4 rows where children is null
data.dropna(subset=['children'], inplace=True)

# NaN in agent/company means 'no agent' / 'no company' — fill with 0
data['agent'].fillna(0, inplace=True)
data['company'].fillna(0, inplace=True)

# Group rare countries under 'Other'
top_countries = data['country'].value_counts().nlargest(20).index
data['country'] = data['country'].apply(lambda x: x if x in top_countries else 'Other')
data['country'].fillna('Unknown', inplace=True)

# One-hot encode nominal (unordered) categorical columns
nominal_cols = [
    'hotel', 'meal', 'country', 'market_segment', 'distribution_channel',
    'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type'
]
data = pd.get_dummies(data, columns=nominal_cols, drop_first=False)

# Encode month as an integer (January=1 ... December=12) — ordinal is appropriate here
month_map = {'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,
             'July':7,'August':8,'September':9,'October':10,'November':11,'December':12}
data['arrival_date_month'] = data['arrival_date_month'].map(month_map)

print(f'Preprocessed shape: {data.shape}')
print(f'Features available: {data.shape[1] - 2} (excluding is_canceled and adr)')

## 4. Train/Test Split

We create two separate feature matrices from the same preprocessed data:

- **Classification task** — target is `is_canceled`, `adr` is a feature
- **Regression task** — target is `adr`, `is_canceled` is excluded (to avoid data leakage)

Both use a 75/25 stratified split with the same random seed as the classification notebook, so results are comparable.

In [ ]:
# ── Classification split (target = is_canceled) ───────────────────────────────
X_clf = data.drop(columns=['is_canceled'])
y_clf = data['is_canceled']

X_train, X_test, y_train, y_test = train_test_split(
    X_clf, y_clf, test_size=0.25, random_state=RANDOM_STATE, stratify=y_clf
)
print(f'Classification — Train: {X_train.shape[0]:,}  Test: {X_test.shape[0]:,}')

# ── Regression split (target = adr) ──────────────────────────────────────────
# Remove extreme ADR outliers (above 99th percentile) — likely data entry errors
adr_series = data['adr']
mask_adr   = adr_series < adr_series.quantile(0.99)
X_reg = data[mask_adr].drop(columns=['is_canceled', 'adr'])
y_reg = data[mask_adr]['adr']

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=RANDOM_STATE
)
print(f'Regression    — Train: {X_reg_train.shape[0]:,}  Test: {X_reg_test.shape[0]:,}')
print(f'Mean ADR (train): ${y_reg_train.mean():.2f}  |  Std: ${y_reg_train.std():.2f}')

## 5. Train the Cancellation Classifier

We train a LightGBM classifier to get **P(cancel)** for every booking. This probability is the core input to the overbooking optimizer — without it we'd have no way to estimate how many guests will actually show up on a given night.

We use the same LightGBM configuration as in the main classification notebook, which achieved strong AUC performance on this dataset.

In [ ]:
lgbm_clf = LGBMClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=8,
    num_leaves=63, subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
)
lgbm_clf.fit(X_train, y_train)

p_cancel = lgbm_clf.predict_proba(X_test)[:, 1]   # P(cancel) for each test booking
p_show   = 1 - p_cancel                            # P(show up)

auc = roc_auc_score(y_test, p_cancel)
print(f'Classifier AUC on test set: {auc:.4f}')
print(f'Average P(cancel) on test set: {p_cancel.mean():.3f}')
print(f'Average P(show up) on test set: {p_show.mean():.3f}')

## 6. ADR Regression — Dynamic Pricing

The `adr` (average daily rate) column records the rate that was charged for each booking. By training a regression model on the same booking features, we can learn the relationship between booking characteristics and price — and use it to suggest optimal rates for new bookings.

This is a simplified form of **dynamic pricing**: rather than using a fixed rate, the hotel adjusts price based on who is booking, when, through which channel, and for how long.

We compare two models:
- **Linear Regression** — interpretable baseline; assumes a linear relationship between features and rate
- **LightGBM Regressor** — captures non-linear interactions (e.g., rates that spike for certain room types during peak months)

We evaluate both on RMSE (lower = better) and R² (higher = better, max 1.0).

In [ ]:
# ── Linear Regression baseline ────────────────────────────────────────────────
# Linear regression requires scaled features because it's sensitive to feature magnitude.
scaler_reg     = StandardScaler()
X_reg_train_s  = scaler_reg.fit_transform(X_reg_train)
X_reg_test_s   = scaler_reg.transform(X_reg_test)

lr_reg  = LinearRegression()
lr_reg.fit(X_reg_train_s, y_reg_train)
lr_pred = lr_reg.predict(X_reg_test_s)

lr_rmse = np.sqrt(mean_squared_error(y_reg_test, lr_pred))
lr_r2   = r2_score(y_reg_test, lr_pred)
print(f'Linear Regression  |  RMSE: ${lr_rmse:.2f}  |  R²: {lr_r2:.4f}')

# ── LightGBM Regressor ────────────────────────────────────────────────────────
# Tree-based models work on raw (unscaled) features.
lgbm_reg  = LGBMRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=8,
    num_leaves=63, subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
)
lgbm_reg.fit(X_reg_train, y_reg_train)
lgbm_pred = lgbm_reg.predict(X_reg_test)

lgbm_rmse = np.sqrt(mean_squared_error(y_reg_test, lgbm_pred))
lgbm_r2   = r2_score(y_reg_test, lgbm_pred)
print(f'LightGBM Regressor |  RMSE: ${lgbm_rmse:.2f}  |  R²: {lgbm_r2:.4f}')

### Visualizing ADR Prediction Quality

Two charts help us evaluate the regression model:

- **Actual vs. Predicted** — if the model were perfect, all points would fall on the red diagonal line. Scatter around it tells us how much uncertainty remains.
- **Residual Distribution** — residuals (actual − predicted) should be centered around zero. A strongly skewed residual distribution would indicate systematic bias in one direction.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted (sample 2,000 points to keep the plot readable)
sample_idx = np.random.choice(len(y_reg_test), size=min(2000, len(y_reg_test)), replace=False)
axes[0].scatter(y_reg_test.iloc[sample_idx], lgbm_pred[sample_idx],
                alpha=0.3, s=10, color='steelblue')
lims = [y_reg_test.min(), y_reg_test.quantile(0.99)]
axes[0].plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual ADR ($)')
axes[0].set_ylabel('Predicted ADR ($)')
axes[0].set_title(f'LightGBM: Actual vs Predicted ADR\nRMSE=${lgbm_rmse:.2f}  R²={lgbm_r2:.3f}')
axes[0].legend()

# Residual distribution
residuals = y_reg_test.values - lgbm_pred
axes[1].hist(residuals, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5, label='Zero error')
axes[1].axvline(residuals.mean(), color='orange', linestyle='-', linewidth=1.5,
                label=f'Mean error: ${residuals.mean():.2f}')
axes[1].set_xlabel('Residual ($)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution\n(centered near 0 = unbiased predictions)')
axes[1].legend()

plt.suptitle('ADR Regression — Dynamic Pricing Model', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Stochastic Overbooking Optimizer

### The Problem

Every night, a hotel has a fixed number of rooms. Guests who booked may or may not show up — and we now have a model that tells us the probability of each outcome. The hotel faces a classic tradeoff:

- If it accepts only as many bookings as it has rooms, cancelled guests leave rooms empty → **lost revenue**
- If it overbooks by too many rooms and more guests show up than expected, some must be walked (relocated to another hotel) → **walkout cost + reputation damage**

The question is: **what is the optimal number of extra bookings `k` to accept?**

### The Mathematical Framework — Newsvendor Model

This is a classical problem in operations research called the **Newsvendor problem**. We model the number of arrivals as a random variable and find the `k` that maximizes expected net revenue.

**Inputs:**
- `p_i` = P(cancel) for booking i, from our LightGBM classifier
- `C` = hotel room capacity
- `Cu` = underage cost = revenue lost per empty room
- `Co` = overage cost = walkout penalty per walked guest

**Expected arrivals per night:**
$$\mu = \sum_i (1 - p_i) \qquad \sigma^2 = \sum_i p_i(1 - p_i)$$

**Optimal overbooking level (Newsvendor critical ratio):**
$$k^* = F^{-1}\!\left(\frac{C_u}{C_u + C_o}\right) - C$$

where $F$ is the CDF of the Normal approximation to the arrival distribution.

We then **validate this analytical result using Monte Carlo simulation** — simulating 5,000 random nights and measuring actual net revenue at each overbooking level.

### Step 1 — Build Per-Night Arrival Distributions

We group test-set bookings by their arrival week and compute, for each night:
- **Expected arrivals** (μ): sum of P(show up) across all bookings for that night
- **Variance of arrivals** (σ²): sum of P(show) × P(cancel) — this captures the uncertainty

We filter to nights with at least 20 bookings so our Normal approximation is stable.

In [ ]:
# Group test bookings by arrival year-week to simulate individual nights
arrival_dates = (df.loc[X_test.index, 'arrival_date_year'].astype(str) + '-W' +
                 df.loc[X_test.index, 'arrival_date_week_number'].astype(str))

night_df = pd.DataFrame({
    'date':   arrival_dates.values,
    'p_show': p_show,
    'adr':    df.loc[X_test.index, 'adr'].values,
    'stay':   (df.loc[X_test.index, 'stays_in_week_nights'] +
               df.loc[X_test.index, 'stays_in_weekend_nights']).values
})
night_df['stay'] = night_df['stay'].clip(lower=1)  # at least 1 night

nights = night_df.groupby('date').agg(
    n_bookings   = ('p_show', 'count'),
    mu_arrivals  = ('p_show', 'sum'),                              # E[arrivals]
    var_arrivals = ('p_show', lambda x: (x * (1 - x)).sum()),     # Var[arrivals]
    avg_adr      = ('adr',    'mean')
).reset_index()

# Keep only nights with enough bookings for a stable Normal approximation
nights = nights[nights['n_bookings'] >= 20].copy()
nights['sigma_arrivals'] = np.sqrt(nights['var_arrivals'])

mu_avg    = nights['mu_arrivals'].mean()
sigma_avg = nights['sigma_arrivals'].mean()

print(f'Nights analyzed: {len(nights)}')
print(f'Avg bookings per night:   {nights["n_bookings"].mean():.1f}')
print(f'Avg expected arrivals:    {mu_avg:.1f}  (μ)')
print(f'Avg arrival std dev:      {sigma_avg:.1f}  (σ)')

### Step 2 — Compute the Optimal Overbooking Level (Newsvendor)

We set the hotel capacity `C` equal to average expected nightly arrivals — this is the no-overbooking baseline. We then compute the **critical ratio** and invert the Normal CDF to find `k*`.

The walkout penalty `W` is set to **\$250** as a default. We'll explore how this assumption affects the result in the sensitivity analysis.

In [ ]:
# Hotel parameters
C        = int(mu_avg)               # room capacity = avg expected arrivals
adr_avg  = df['adr'].median()        # representative nightly rate
avg_stay = max((df['stays_in_week_nights'] + df['stays_in_weekend_nights']).median(), 1)
W        = 250                       # walkout penalty per walked guest ($)

Cu = adr_avg * avg_stay              # underage cost: revenue lost per empty room
Co = W                               # overage cost: penalty per walked guest

critical_ratio = Cu / (Cu + Co)

# Optimal k: inverse Normal CDF at the critical ratio, shifted by capacity
k_star = max(0, int(norm.ppf(critical_ratio, loc=mu_avg, scale=sigma_avg) - C))

print('─' * 45)
print(f'Hotel capacity C:            {C} rooms')
print(f'Avg Daily Rate (median):     ${adr_avg:.2f}')
print(f'Avg stay length:             {avg_stay:.1f} nights')
print(f'Underage cost Cu:            ${Cu:.2f}  (empty room)')
print(f'Overage cost Co (W):         ${Co:.2f}  (walked guest)')
print(f'Critical ratio:              {critical_ratio:.3f}')
print(f'Optimal overbooking k*:      {k_star} rooms')
print('─' * 45)

### Step 3 — Monte Carlo Simulation

The Newsvendor formula gives us an analytical answer, but it rests on a Normal approximation of arrivals. We validate and enrich this result by **simulating 5,000 random nights** at every overbooking level from `k=0` to `k=20`.

For each simulated night:
1. Draw actual arrivals from `Normal(μ, σ)` — representing the randomness in who actually shows up
2. Compute revenue: `min(arrivals, C + k) × adr_avg × avg_stay`
3. Compute walkout cost: `max(arrivals - C - k, 0) × W`
4. Net revenue = revenue − walkout cost

At the end, we have a full distribution of net revenue for each `k`, not just a point estimate.

In [ ]:
N_SIM    = 5000
k_values = range(0, 21)
mc_results = {}

for k in k_values:
    total_cap = C + k
    net_revenues = []
    for _ in range(N_SIM):
        # Random arrivals drawn from Normal approximation of Poisson-Binomial
        arrivals = max(0, int(np.round(np.random.normal(mu_avg, sigma_avg))))

        checked_in = min(arrivals, total_cap)
        walked     = max(arrivals - total_cap, 0)

        revenue      = checked_in * adr_avg * avg_stay
        walkout_cost = walked * W
        net_revenues.append(revenue - walkout_cost)

    mc_results[k] = np.array(net_revenues)

mean_net = {k: mc_results[k].mean() for k in k_values}
std_net  = {k: mc_results[k].std()  for k in k_values}

best_k_mc = max(mean_net, key=mean_net.get)
print(f'Monte Carlo best k: {best_k_mc}  (Newsvendor k*: {k_star})')
print(f'Revenue gain over no-overbook: ${mean_net[best_k_mc] - mean_net[0]:,.2f}/night')

### Visualizing the Optimizer Results

Two charts summarize the simulation:

- **Left** — expected net revenue at each overbooking level, with the ±1 std band showing night-to-night variability. The gold dashed line marks the optimal `k*`. Revenue rises as we overbook more (filling rooms that would have been cancelled), peaks, then falls as walkout costs dominate.
- **Right** — the full revenue distribution at `k=0` vs. `k*`. The rightward shift of the blue distribution shows the expected gain from optimal overbooking.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ks    = list(k_values)
means = [mean_net[k] for k in ks]
stds  = [std_net[k]  for k in ks]

# Revenue curve
axes[0].plot(ks, means, 'o-', color='steelblue', linewidth=2, markersize=6,
             label='Expected net revenue')
axes[0].fill_between(ks,
                     [m - s for m, s in zip(means, stds)],
                     [m + s for m, s in zip(means, stds)],
                     alpha=0.15, color='steelblue', label='±1 std dev')
axes[0].axvline(x=k_star, color='gold',   linewidth=2.5, linestyle='--',
                label=f'Optimal k* = {k_star} (Newsvendor)')
axes[0].axvline(x=0,      color='tomato', linewidth=1.5, linestyle=':',
                label='No overbooking (k=0)')
axes[0].set_xlabel('Overbooking level k (extra rooms accepted)', fontsize=11)
axes[0].set_ylabel('Expected Net Revenue per Night ($)', fontsize=11)
axes[0].set_title('Overbooking Optimizer\nExpected Revenue vs. k', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Revenue distribution: no-overbook vs optimal
axes[1].hist(mc_results[0],      bins=60, alpha=0.6, color='tomato',
             label=f'k=0 (no overbook)  avg=${mean_net[0]:,.0f}', density=True)
axes[1].hist(mc_results[k_star], bins=60, alpha=0.6, color='steelblue',
             label=f'k={k_star} (optimal)  avg=${mean_net[k_star]:,.0f}', density=True)
axes[1].axvline(mean_net[0],      color='tomato',    linewidth=2, linestyle='--')
axes[1].axvline(mean_net[k_star], color='steelblue', linewidth=2, linestyle='--')
axes[1].set_xlabel('Net Revenue per Night ($)', fontsize=11)
axes[1].set_ylabel('Density', fontsize=11)
axes[1].set_title(f'Revenue Distribution: k=0 vs k={k_star}\n'
                  f'Expected nightly gain = ${mean_net[k_star]-mean_net[0]:,.0f}', fontsize=12)
axes[1].legend(fontsize=9)

plt.suptitle('Stochastic Overbooking Optimizer — Monte Carlo (5,000 simulations)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Revenue at k=0:    ${mean_net[0]:,.2f}/night')
print(f'Revenue at k={k_star}:    ${mean_net[k_star]:,.2f}/night')
print(f'Nightly gain:      ${mean_net[k_star] - mean_net[0]:,.2f}')

## 8. Sensitivity Analysis — How Does k* Change With Walkout Cost?

The walkout penalty `W` is not a fixed number in reality — it depends on the hotel's policies, the market, and compensation offered to walked guests. A budget hotel might pay \$80 to rebook a guest; a luxury hotel might incur \$500 in compensation, rebooking costs, and reputation damage.

This section sweeps `W` from \$50 to \$500 and shows:
- How the optimal overbooking level `k*` decreases as penalties rise (conservative strategy)
- How the expected nightly revenue gain shrinks as walking guests becomes more costly

The output includes a **decision table** management can use directly: look up your walkout cost and read off the recommended overbooking buffer.

In [ ]:
W_values   = np.arange(50, 510, 10)
k_star_W   = []
rev_gain_W = []

for W_val in W_values:
    # Recompute optimal k* for this walkout cost
    cr  = Cu / (Cu + W_val)
    k_s = max(0, int(norm.ppf(cr, loc=mu_avg, scale=sigma_avg) - C))
    k_star_W.append(k_s)

    # Estimate revenue gain via 1,000 Monte Carlo simulations (faster than 5,000)
    gains = []
    for _ in range(1000):
        arr    = max(0, int(np.round(np.random.normal(mu_avg, sigma_avg))))
        rev_k  = min(arr, C + k_s) * adr_avg * avg_stay - max(arr - C - k_s, 0) * W_val
        rev_0  = min(arr, C) * adr_avg * avg_stay
        gains.append(rev_k - rev_0)
    rev_gain_W.append(np.mean(gains))

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(W_values, k_star_W, 'o-', color='steelblue', markersize=4, linewidth=2)
axes[0].axvline(x=250, color='gold', linestyle='--', linewidth=1.5, label='Default W=$250')
axes[0].set_xlabel('Walkout Penalty per Guest W ($)', fontsize=11)
axes[0].set_ylabel('Optimal Overbooking k*', fontsize=11)
axes[0].set_title('Optimal Overbooking vs. Walkout Cost\n(lower penalty → accept more overbookings)', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].plot(W_values, rev_gain_W, 'o-', color='green', markersize=4, linewidth=2)
axes[1].axhline(y=0, color='gray', linestyle='--', linewidth=1)
axes[1].axvline(x=250, color='gold', linestyle='--', linewidth=1.5, label='Default W=$250')
axes[1].set_xlabel('Walkout Penalty per Guest W ($)', fontsize=11)
axes[1].set_ylabel('Expected Nightly Revenue Gain ($)', fontsize=11)
axes[1].set_title('Revenue Gain from Optimal Overbooking\n(diminishing returns as W rises)', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Sensitivity Analysis — Walkout Cost vs. Optimal Strategy',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Decision table
print('\nDecision Table: Optimal Overbooking Level by Walkout Penalty')
print('─' * 48)
print(f'{"Walkout Cost W":>16}  {"Optimal k*":>10}  {"Nightly Gain":>14}')
print('─' * 48)
for W_val, k_s, gain in zip(W_values[::5], k_star_W[::5], rev_gain_W[::5]):
    print(f'  ${W_val:>12.0f}  {k_s:>10}  ${gain:>12.2f}')
print('─' * 48)

## 9. Summary

### What We Built

| Model | Output | Use |
|---|---|---|
| LightGBM Classifier | P(cancel) per booking | Input to optimizer |
| LightGBM Regressor | Predicted ADR | Dynamic pricing |
| Newsvendor + Monte Carlo | Optimal k* rooms to overbook | Revenue management decision |
| Sensitivity analysis | k* as a function of walkout cost W | Customizable policy table |

### Key Takeaways

1. **The cancellation model is the foundation.** Without P(cancel), the optimizer would have no way to estimate expected arrivals — it would be working with averages rather than per-booking uncertainty.

2. **Overbooking has a clear optimal point.** Revenue rises as `k` increases (filling cancelled rooms), but falls sharply once walkout costs dominate. The Newsvendor formula and Monte Carlo both converge on the same `k*`, which is reassuring.

3. **The right strategy depends on your walkout cost.** A hotel that negotiates cheap rebooking deals can afford to overbook more aggressively. One with strict guest-satisfaction standards should be more conservative. The sensitivity table makes this relationship explicit.

4. **ADR regression has room to grow.** The LightGBM regressor outperforms linear regression, but ADR is also driven by market dynamics (competitor pricing, seasonality, local events) that aren't in this dataset. A production pricing model would incorporate those signals.

### Next Steps

- Add real room capacity data (this notebook uses estimated average arrivals as a proxy for C)
- Incorporate room type differentiation — optimize separately per room category
- Extend to a multi-period LP: optimize overbooking policy across a full booking horizon, not just one night